# Environment / Dependencies Setup
Installs LangGraph (agent orchestration), the LangChain LLM/embedding wrappers, PDF/download helpers, the Neo4j stack (Part 2), and dotenv.

In [ ]:
!pip install -qU langgraph langchain-openai langchain-huggingface langchain-text-splitters pypdf requests scikit-learn neo4j yfiles-jupyter-graphs python-dotenv

## Setup — Load the Document Once

### Step 1 — Imports

In [ ]:
import os
import json
import requests
from typing import TypedDict, List
from dotenv import load_dotenv
from pypdf import PdfReader
from neo4j import GraphDatabase
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import ChatOpenAI
from langchain_huggingface import HuggingFaceEmbeddings
from sklearn.metrics.pairwise import cosine_similarity
from langgraph.graph import StateGraph, START, END
from IPython.display import Image, display

# Load the secrets (OPENROUTER_API_KEY, NEO4J_*) from the .env file next to this notebook
load_dotenv(".env")


All the imports for both parts sit here, so later steps only contain the new ideas.


### Step 2 — Configure the LLM and the Embedding Model

In [ ]:
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")

if not OPENROUTER_API_KEY:
    OPENROUTER_API_KEY = input("Enter your OpenRouter API key (get one at https://openrouter.ai): ").strip()

llm = ChatOpenAI(
    openai_api_key=OPENROUTER_API_KEY,
    openai_api_base="https://openrouter.ai/api/v1",
    model_name="nvidia/nemotron-3-super-120b-a12b:free",
    temperature=0.0,
    extra_body={"reasoning": {"enabled": False}},
)

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
print("LLM and embedding model configured successfully!")

# Setup — LLM and Embedding Model
One LLM connection serves every thinking step (grade, rewrite, route, answer). `temperature=0.0` keeps the YES/NO grade and the router's choice consistent, and hidden reasoning is disabled on both models. `all-MiniLM-L6-v2` embeds locally with no key.

In [ ]:
os.makedirs("data", exist_ok=True)

PDF_URL = "https://arxiv.org/pdf/1706.03762.pdf"
PDF_FILENAME = "data/downloaded_paper.pdf"

print(f"Downloading PDF from {PDF_URL}...")
response = requests.get(PDF_URL)
response.raise_for_status()

with open(PDF_FILENAME, "wb") as f:
    f.write(response.content)
print(f"PDF downloaded to {PDF_FILENAME}!")

reader = PdfReader(PDF_FILENAME)
document_text = ""
for page in reader.pages:
    document_text += page.extract_text() + "\n"
    if len(document_text) >= 1500:
        document_text = document_text[:1500]
        break

print(f"Extracted {len(document_text)} characters of text.")

### Step 3 — Download the PDF and Extract Text
Creates `data/` and gathers `document_text` — the one document both parts use.

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50,
    separators=["\n\n", "\n", ".", " ", ""]
)
knowledge_base = text_splitter.split_text(document_text)
print(f"Knowledge base ready with {len(knowledge_base)} chunks.")

knowledge_vectors = embeddings.embed_documents(knowledge_base)
print(f"Embedded {len(knowledge_vectors)} chunks ({len(knowledge_vectors[0])} numbers each).")

### Step 4 — Split into Chunks and Embed Them
Paragraph-first splitter with 50-char overlap; `knowledge_vectors` holds one 384-dim vector per chunk in the same order as `knowledge_base`.

In [ ]:
class AgentState(TypedDict):
    question: str               # the current question (may get rewritten)
    retrieved_facts: List[str]  # chunks found so far
    grade: str                  # "YES" or "NO" from the grading step
    retry_count: int            # how many times we have tried again
    final_answer: str           # filled in only at the end


def new_state(question: str, **extra) -> dict:
    """A fresh starting state for one question."""
    return {"question": question, "retrieved_facts": [], "grade": "",
            "retry_count": 0, "final_answer": "", **extra}

### Step 5 — Define the Agent's State
The shared notebook every node reads and writes; `new_state` builds a fresh one so each run starts clean.

In [ ]:
def vector_search(question: str, top_k: int = 2) -> List[str]:
    """Return the top_k chunks whose meaning is closest to the question."""
    question_vector = embeddings.embed_query(question)
    scores = cosine_similarity([question_vector], knowledge_vectors)[0]
    top_indices = scores.argsort()[-top_k:][::-1]   # best match first
    return [knowledge_base[i] for i in top_indices]


def retrieve_node(state: AgentState) -> AgentState:
    print(f"Retrieving for: '{state['question']}'")
    state["retrieved_facts"] = vector_search(state["question"])
    return state

### Step 6 — Retrieve Node (Vector Search)
Embeds the question, scores every chunk with cosine similarity, and keeps the best two.

In [ ]:
def grade_node(state: AgentState) -> AgentState:
    facts_text = "\n".join(state["retrieved_facts"])

    prompt = f"""
    You are a grading assistant evaluating search results.

    Question: {state['question']}
    Retrieved facts:
    {facts_text}

    Do these facts contain ANY relevant hints, keywords, or partial information that could help answer the question?
    Reply with ONLY one word: YES or NO.
    """

    raw_grade = llm.invoke(prompt).content.strip().upper()
    state["grade"] = "YES" if "YES" in raw_grade else "NO"
    print(f"Grade: {state['grade']}")
    return state

### Step 7 — Grade Node
Asks for any helpful hint so partly useful chunks aren't rejected; `"YES" in raw_grade` tolerates extra punctuation.

In [ ]:
def rewrite_node(state: AgentState) -> AgentState:
    prompt = f"""
    This question did not return good search results: "{state['question']}"
    Rewrite it to be clearer and easier to search for. Reply with only the new question.
    """

    state["question"] = llm.invoke(prompt).content.strip()
    state["retry_count"] += 1
    print(f"Rewritten question: {state['question']}")
    return state

### Step 8 — Rewrite Node
Runs only after a `NO` grade: replaces the question with the LLM's rewrite and increments `retry_count`.

In [ ]:
def generate_node(state: AgentState) -> AgentState:
    facts_text = "\n".join(state["retrieved_facts"])

    prompt = f"""
    Answer the question using ONLY these facts. If the facts do not contain the answer, say that the document does not cover it.
    {facts_text}

    Question: {state['question']}

    Output your response in EXACTLY two sections:
    --- FINAL ANSWER ---
    [Give a short, direct answer.]

    --- EXPLAINABILITY ---
    [Briefly summarize the specific facts or paths provided above that you used to construct this answer.]
    """

    state["final_answer"] = llm.invoke(prompt).content.strip()
    return state

### Step 9 — Generate Node
The only node that writes the answer — reached on a `YES` grade or at the retry limit — including an `EXPLAINABILITY` section.

In [ ]:
def decide_after_grading(state: AgentState) -> str:
    if state["grade"] == "YES" or state["retry_count"] >= 2:
        return "generate"
    return "rewrite"


builder = StateGraph(AgentState)
builder.add_node("retrieve", retrieve_node)
builder.add_node("grade", grade_node)
builder.add_node("rewrite", rewrite_node)
builder.add_node("generate", generate_node)

builder.add_edge(START, "retrieve")
builder.add_edge("retrieve", "grade")
# keys are the labels decide_after_grading returns, values are the nodes they jump to
builder.add_conditional_edges("grade", decide_after_grading, {
    "generate": "generate",
    "rewrite": "rewrite",
})
builder.add_edge("rewrite", "retrieve")
builder.add_edge("generate", END)

agent = builder.compile()
print("Self-correcting agent compiled successfully!")

try:
    display(Image(agent.get_graph().draw_mermaid_png()))
except Exception:
    print(agent.get_graph().draw_mermaid())

### Step 10 — Routing Function and Graph
`decide_after_grading` just picks the next node (`retry_count >= 2` is the limit); `add_conditional_edges` wires the two destinations.

In [ ]:
result = agent.invoke(new_state("How does the Transformer work?"))

print(f"Retries used: {result['retry_count']}")
print(result["final_answer"])

### Step 11 — Run the Agent
`agent.invoke` starts at `START` and loops through `rewrite` until `END`. You should see `Grade: YES`, `Retries used: 0`, and a two-section answer.

In [ ]:
result = agent.invoke(new_state("What is the boiling point of liquid nitrogen?"))

print(f"Retries used: {result['retry_count']}")
print(result["final_answer"])

### Step 12 — Force a Retry
A question the paper can't answer grades `NO`, gets rewritten, and after two rewrites the retry limit forces an answer with what it has.

Try it yourself: the rewrite cannot rescue this question because the paper has no answer. Ask a vague question the paper can answer, such as "transformer stuff", and note whether the grade or rewrite changes.

In [ ]:
# Values come from the .env file loaded in Step 1
NEO4J_URI = os.getenv("NEO4J_URI")
NEO4J_USERNAME = os.getenv("NEO4J_USERNAME")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD")

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))
driver.verify_connectivity()
print("Connected to Neo4j successfully!")


### Step 13 — Connect to Neo4j
Values come from the `.env` file loaded in Step 1; `verify_connectivity()` fails right here if they are wrong.
Part 2 needs a (free) Neo4j Aura instance; a driver is the connection object and a session is one short conversation with the database.

### Step 14 — Build the Knowledge Graph
**Warning:** the next cell runs `MATCH (n) DETACH DELETE n`, which deletes every node and relationship in the connected Neo4j database. Use a fresh or throwaway Aura instance, never one holding data you want to keep.

In [ ]:
with driver.session() as session:
    session.run("MATCH (n) DETACH DELETE n")
print("Cleared existing Neo4j database.")

extract_prompt = f"""
Extract key concepts and their relationships from this text.
Return ONLY a valid JSON object with format:
{{
  "nodes": ["Concept1", "Concept2"],
  "relationships": [
    {{"source": "Concept1", "target": "Concept2", "type": "RELATIONSHIP_TYPE"}}
  ]
}}

Text:
{document_text}
"""
raw_json = llm.invoke(extract_prompt).content.strip()
raw_json = raw_json[raw_json.find("{"): raw_json.rfind("}") + 1]   # drop any markdown code fences
graph_data = json.loads(raw_json)
print(f"LLM extracted {len(graph_data['nodes'])} nodes and {len(graph_data['relationships'])} relationships.")

Slicing from the first `{` to the last `}` removes any markdown code fences some models wrap around JSON. Now write the result into Neo4j:

In [ ]:
with driver.session() as session:
    for node in graph_data["nodes"]:
        session.run("MERGE (c:Concept {name: $name})", name=node)

    for rel in graph_data["relationships"]:
        rel_type = rel["type"].replace(" ", "_").upper()
        # backticks let Cypher accept relationship names with unusual characters
        session.run(
            f"MATCH (a:Concept {{name: $source}}), (b:Concept {{name: $target}}) "
            f"MERGE (a)-[r:`{rel_type}`]->(b)",
            source=rel["source"], target=rel["target"],
        )
print("Graph written into Neo4j.")

Slicing from the first `{` to the last `}` strips any markdown fences around the JSON.

In [ ]:
with driver.session() as session:
    # 384 must match the embedding model's output size (all-MiniLM-L6-v2)
    session.run("""
    CREATE VECTOR INDEX concept_embeddings IF NOT EXISTS
    FOR (c:Concept) ON (c.embedding)
    OPTIONS {indexConfig: {`vector.dimensions`: 384, `vector.similarity_function`: 'cosine'}}
    """)

    names = [record["name"] for record in session.run("MATCH (c:Concept) RETURN c.name AS name")]
    for name in names:
        session.run("MATCH (c:Concept {name: $name}) SET c.embedding = $vector",
                    name=name, vector=embeddings.embed_query(name))

    n_rels = session.run("MATCH ()-[r]->() RETURN count(r) AS n").single()["n"]
print(f"Neo4j now holds {len(names)} concepts, {n_rels} relationships, and a vector index.")

`MERGE` creates each node or relationship only if it does not exist yet. Then every concept is embedded and a vector index is built so the graph tool can match concepts to the question. Expect roughly a dozen concepts; the stored relationship count can be lower than the number the LLM returned, because `MERGE` collapses duplicates and links to unknown names are skipped.

In [ ]:
# the vector index seeds 2 concepts, then -[r]- follows their links in either direction
CYPHER_GRAPH = """
CALL db.index.vector.queryNodes('concept_embeddings', 2, $vector)
YIELD node AS seed, score
MATCH (seed)-[r]-(neighbor:Concept)
RETURN seed.name + ' --[' + type(r) + ']-> ' + neighbor.name AS path
LIMIT 5
"""


def graph_search(question: str) -> List[str]:
    """Find the 2 concepts closest to the question, then follow their links."""
    question_vector = embeddings.embed_query(question)
    with driver.session() as session:
        paths = [record["path"] for record in session.run(CYPHER_GRAPH, vector=question_vector)]
    return paths or ["No relevant information found in Neo4j."]

### Step 15 — The Graph Tool
The vector index finds the seed concepts closest to the question, then `MATCH (seed)-[r]-(neighbor)` walks one step along each link. The pattern has no arrowhead, so links match in either direction; the `->` in each printed line is only a separator, not the stored direction.

In [ ]:
class RoutedState(AgentState):
    current_tool: str           # "VECTOR" or "GRAPH"


def router_node(state: RoutedState) -> RoutedState:
    prompt = f"""
    You are an intelligent router. Analyze this question: "{state['question']}"

    Which retrieval tool is best suited?
    - VECTOR: Best for general concepts, definitions, or broad semantic meaning.
    - GRAPH: Best for connections, architectural components, relationships, or multi-hop logic.

    Reply with ONLY one word: VECTOR or GRAPH.
    """

    chosen = llm.invoke(prompt).content.strip().upper()
    state["current_tool"] = "GRAPH" if "GRAPH" in chosen else "VECTOR"
    print(f"ROUTER DECISION: Using {state['current_tool']} tool.")
    return state

### Step 16 — Router Node and the Routed State
`RoutedState` is Part 1's state plus one field, so all Part 1 nodes are reused unchanged. The router picks the tool and defaults to VECTOR.

In [ ]:
def routed_retrieve_node(state: RoutedState) -> RoutedState:
    print(f"Retrieving for: '{state['question']}' via {state['current_tool']} tool")
    if state["current_tool"] == "GRAPH":
        state["retrieved_facts"] = graph_search(state["question"])
    else:
        state["retrieved_facts"] = vector_search(state["question"])
    return state


def fallback_node(state: RoutedState) -> RoutedState:
    if state["retry_count"] == 0:
        # First failure: maybe the wrong tool was picked, so swap it
        old_tool = state["current_tool"]
        state["current_tool"] = "VECTOR" if old_tool == "GRAPH" else "GRAPH"
        state["retry_count"] += 1
        print(f"Fallback: swapping tool from {old_tool} to {state['current_tool']}")
    else:
        # Second failure: the question itself is the problem, so reuse Part 1's rewrite
        state = rewrite_node(state)
    return state

### Step 17 — Routed Retrieve Node and Fallback Node
`routed_retrieve_node` calls whichever tool the router chose; `fallback_node` swaps the tool first, then rewrites the question.

In [ ]:
builder = StateGraph(RoutedState)
builder.add_node("router", router_node)
builder.add_node("retrieve", routed_retrieve_node)
builder.add_node("grade", grade_node)
builder.add_node("fallback", fallback_node)
builder.add_node("generate", generate_node)

builder.add_edge(START, "router")
builder.add_edge("router", "retrieve")
builder.add_edge("retrieve", "grade")
# same grading function as Part 1, but its "rewrite" label now targets the fallback node
builder.add_conditional_edges("grade", decide_after_grading, {
    "generate": "generate",
    "rewrite": "fallback",
})
builder.add_edge("fallback", "retrieve")
builder.add_edge("generate", END)

routed_agent = builder.compile()
print("Routed agent compiled successfully!")

try:
    display(Image(routed_agent.get_graph().draw_mermaid_png()))
except Exception:
    print(routed_agent.get_graph().draw_mermaid())

### Step 18 — Build the Routed Graph
Reuses the same `decide_after_grading`; its `"rewrite"` answer points at the `fallback` node, which loops back into `retrieve`.

In [ ]:
questions = [
    "What is the Transformer?",
    "What models or mechanisms are sequence transduction models based on or connected to?",
    "What is the boiling point of liquid nitrogen?",
]

for q in questions:
    print("=" * 70)
    # current_tool must be present in the state; the router overwrites it immediately
    result = routed_agent.invoke(new_state(q, current_tool=""))
    print(f"Tool at the end: {result['current_tool']} | Retries used: {result['retry_count']}")
    print(result["final_answer"])

### Step 19 — Run Three Questions
Meaning question -> VECTOR; connection question -> GRAPH; unanswerable question -> tool swap, then rewrite, then an answer at the retry limit.

In [ ]:
from yfiles_jupyter_graphs import GraphWidget

with driver.session() as session:
    graph_result = session.run("MATCH (n)-[r]->(m) RETURN n, r, m")
    # Build the widget while the session is still open: the result is consumed once it closes
    widget = GraphWidget(graph=graph_result.graph())

display(widget)
driver.close()

### Step 20 — Visualize the Graph
Draws the concepts the graph tool walks through, then closes the driver.

### What We Learnt
A plain pipeline became an agent that grades its retrieval, chooses between two search tools, and recovers when its first choice fails.